# FACODI Workspace Setup
Execute this notebook to set up the FACODI operational workspace in Odoo.

In [1]:
import os, sys, json
from pathlib import Path
os.environ.pop('SSLKEYLOGFILE', None)

workspace_dir = Path().cwd()
workspace_odoo = workspace_dir / 'workspace' / 'odoo'
sys.path.insert(0, str(workspace_odoo))

print(f'Working directory: {workspace_dir}')
print(f'Python path includes: {workspace_odoo}')

Working directory: c:\Users\marce\Desktop\Workspace\Monynha Softwares\Projects\On live\facodi.pt
Python path includes: c:\Users\marce\Desktop\Workspace\Monynha Softwares\Projects\On live\facodi.pt\workspace\odoo


In [2]:
from odoo_test_utils import load_env, get_odoo_credentials, OdooClient

workspace_root = workspace_dir / 'workspace'
project_root = workspace_dir
load_env(workspace_root=workspace_root, project_root=project_root)
h, d, u, p = get_odoo_credentials()
c = OdooClient(h, d, u, p)
uid = c.authenticate()
print(f'✓ Authenticated as UID {uid}')

✓ Authenticated as UID 2


In [3]:
# Get or create FACODI project
proj_name = 'FACODI — Digital Platform'
existing = c.execute('project.project', 'search_read', [[('name', '=', proj_name)]], {'fields': ['id']})
if existing:
    proj_id = existing[0]['id']
    print(f'✓ Found project: ID {proj_id}')
else:
    proj_id = c.execute('project.project', 'create', [{'name': proj_name, 'privacy_visibility': 'employees'}])
    print(f'✓ Created project: ID {proj_id}')

✓ Created project: ID 4


In [4]:
# Get users
users = c.execute('res.users', 'search_read', [[('id', '>', 0)]], {'fields': ['id', 'name'], 'limit': 100})
marcelo = next((u['id'] for u in users if 'marcelo' in u['name'].lower()), 2)
bilal = next((u['id'] for u in users if 'bilal' in u['name'].lower()), marcelo)
print(f'✓ Marcelo: {marcelo}')
print(f'✓ Bilal: {bilal}')

✓ Marcelo: 2
✓ Bilal: 7


In [5]:
# Create stages
stages_def = [
    ('Planning', 10),
    ('Definition', 20),
    ('Design / Structure', 30),
    ('Development', 40),
    ('Low-code / Content', 50),
    ('Validation', 60),
    ('Publication', 70),
    ('Iteration', 80),
]

stage_ids = {}
for name, seq in stages_def:
    existing = c.execute('project.task.type', 'search_read', [[('name', '=', name), ('project_ids', 'in', [proj_id])]], {'fields': ['id']})
    if existing:
        sid = existing[0]['id']
    else:
        sid = c.execute('project.task.type', 'create', [{'name': name, 'sequence': seq, 'project_ids': [[4, proj_id]]}])
    stage_ids[name] = sid
    print(f'  ✓ {name}: {sid}')

print(f'\n✓ Created {len(stage_ids)} stages')

  ✓ Planning: 26
  ✓ Definition: 27
  ✓ Design / Structure: 28
  ✓ Development: 29
  ✓ Low-code / Content: 30
  ✓ Validation: 31
  ✓ Publication: 32
  ✓ Iteration: 33

✓ Created 8 stages


In [6]:
# Define task groups
task_groups = [
    ('A. Website Refactor', marcelo, 'Planning', [
        'Audit current website structure and navigation',
        'Review information architecture and current sitemap',
        'Define new sitemap and primary navigation',
        'Design homepage structure and user journey',
        'Define relationship between website and Odoo eLearning',
        'Identify what should remain static vs dynamic',
        'Review CTA structure and conversion paths',
        'Review footer and trust/partner areas',
        'Ensure consistency between brand, mission, and educational offer',
    ]),
    ('B. Course Selection and Expansion', marcelo, 'Definition', [
        'Audit courses already migrated into Odoo',
        'Identify next priority courses to migrate',
        'Define selection criteria for new courses',
        'Group courses by learning path or category',
        'Define MVP course set for public website',
        'Determine which courses are draft/internal vs public-ready',
        'Create planning placeholders for candidate courses',
    ]),
    ('C. Page Planning', bilal, 'Design / Structure', [
        'Plan homepage: objective, audience, main message, CTA',
        'Plan About FACODI: mission, story, team',
        'Plan Courses / Learning Paths: discovery, filtering',
        'Plan Single Course Page template and content structure',
        'Plan Lesson / Content page template',
        'Plan Community page: engagement and discussion',
        'Plan Roadmap page: public transparency',
        'Plan Partners / Institutional Context page',
        'Plan Contact page: support and inquiries',
        'Plan FAQ / How It Works page if justified',
    ]),
    ('D. eLearning and Course Structure', marcelo, 'Design / Structure', [
        'Review and normalize current eLearning structure in Odoo',
        'Define course taxonomy and categories',
        'Define course tags and metadata fields',
        'Define learning path structure and sequencing',
        'Define lesson/content ordering and dependencies',
        'Define public preview and enrollment strategy',
        'Define publication readiness criteria for courses',
        'Audit and complete missing course metadata',
        'Ensure consistency between website pages and course records',
    ]),
    ('E. Content and Copy', bilal, 'Development', [
        'Define page-level copy requirements and messaging',
        'Define content ownership and contribution guidelines',
        'Draft copy backlog for all planned pages',
        'Review messaging consistency across website',
        'Complete missing course descriptions',
        'Add benefit-oriented copy to course listings',
        'Add institutional context and mission explanation',
        'Write CTA copy and call-to-action refinements',
    ]),
    ('F. Technical and Odoo Configuration', marcelo, 'Development', [
        'Review installed modules relevant to FACODI',
        'Verify project management setup and permissions',
        'Verify website module integration and capabilities',
        'Verify eLearning/slide module setup and features',
        'Configure ownership and permission logic',
        'Identify required automations and workflows',
        'Identify and create needed custom fields if any',
        'Document risks and technical blockers',
        'Plan scalable structure for content operations',
    ]),
    ('G. Validation and Publishing Readiness', marcelo, 'Validation', [
        'Validate completeness of all main pages',
        'Validate completeness of primary courses',
        'Test all links and internal consistency',
        'Review task assignments and ownership clarity',
        'Identify and merge duplicated work',
        'Prepare publication checklist and sign-off',
        'Prepare post-launch iteration and improvement list',
        'Coordinate final stakeholder review',
    ]),
]

print(f'Defined {len(task_groups)} task groups')

Defined 7 task groups


In [8]:
# Create task groups and subtasks
total_created = 0
created_groups = []

for group_name, owner, stage, subtasks in task_groups:
    stage_id = stage_ids[stage]
    
    # Create parent task
    parent_id = c.execute('project.task', 'create', [{
        'name': group_name,
        'project_id': proj_id,
        'stage_id': stage_id,
        'user_ids': [[4, owner]],  # Use user_ids (many2many) instead of user_id
    }])
    total_created += 1
    
    # Create subtasks
    for i, sub_name in enumerate(subtasks, 1):
        sub_id = c.execute('project.task', 'create', [{
            'name': f'{i}. {sub_name}',
            'project_id': proj_id,
            'stage_id': stage_id,
            'user_ids': [[4, owner]],  # Use user_ids (many2many) instead of user_id
            'parent_id': parent_id,
        }])
        total_created += 1
    
    created_groups.append({
        'name': group_name,
        'id': parent_id,
        'owner': owner,
        'stage': stage,
        'subtask_count': len(subtasks)
    })
    print(f'✓ {group_name} ({len(subtasks)} subtasks)')

print(f'\n✓✓✓ Created {total_created} total tasks ✓✓✓')

✓ A. Website Refactor (9 subtasks)
✓ B. Course Selection and Expansion (7 subtasks)
✓ C. Page Planning (10 subtasks)
✓ D. eLearning and Course Structure (9 subtasks)
✓ E. Content and Copy (8 subtasks)
✓ F. Technical and Odoo Configuration (9 subtasks)
✓ G. Validation and Publishing Readiness (8 subtasks)

✓✓✓ Created 67 total tasks ✓✓✓


In [9]:
# Verify and save results
result = {
    'status': 'success',
    'project_id': proj_id,
    'project_name': 'FACODI — Digital Platform',
    'stages_created': len(stage_ids),
    'task_groups': created_groups,
    'total_tasks': total_created,
}

print(json.dumps(result, indent=2, default=str))

{
  "status": "success",
  "project_id": 4,
  "project_name": "FACODI \u2014 Digital Platform",
  "stages_created": 8,
  "task_groups": [
    {
      "name": "A. Website Refactor",
      "id": 5,
      "owner": 2,
      "stage": "Planning",
      "subtask_count": 9
    },
    {
      "name": "B. Course Selection and Expansion",
      "id": 15,
      "owner": 2,
      "stage": "Definition",
      "subtask_count": 7
    },
    {
      "name": "C. Page Planning",
      "id": 23,
      "owner": 7,
      "stage": "Design / Structure",
      "subtask_count": 10
    },
    {
      "name": "D. eLearning and Course Structure",
      "id": 34,
      "owner": 2,
      "stage": "Design / Structure",
      "subtask_count": 9
    },
    {
      "name": "E. Content and Copy",
      "id": 44,
      "owner": 7,
      "stage": "Development",
      "subtask_count": 8
    },
    {
      "name": "F. Technical and Odoo Configuration",
      "id": 53,
      "owner": 2,
      "stage": "Development",
      "su